In [1]:
import pandas as pd
import random
import numpy as np

In [2]:
Dropbox_path='~/NYU Langone Health Dropbox/Bianca Cordazzo Vargas/Shenhav_Lab/IMiC'
misame_data_path=f'{Dropbox_path}/Data/MISAME'
vital_data_path=f'{Dropbox_path}/Data/VITAL'
sapient_data_path=f'{Dropbox_path}/Data/Sapient_Box_Data/Raw'
misame_untargeted_out=f'{Dropbox_path}/Code/Joint-RPCA/output/all_tps'
metab_id_path=f'{Dropbox_path}/Code/machine_learning'

In [3]:
print("loading metadata")
misame_full_BM_df_filt=pd.read_csv(f"{misame_data_path}/metadata/misame_processed_metadata.tsv", sep='\t')
misame_traj_df=pd.read_csv(f"{misame_data_path}/metadata/misame_processed_traj_key.csv", sep=',') # from ../Trajectory_analysis/Misame_growth_traj_MISAME.Rmd
misame_full_BM_df_filt=pd.merge(misame_full_BM_df_filt, misame_traj_df, on = 'SubjectID', how = 'left') 

loading metadata


In [4]:
print("loading data")
misame_raw=pd.read_csv(f"{sapient_data_path}/MISAME3_rLC_mtb_raw_data_021224.csv", sep=',', index_col=0)
misame_raw=misame_raw[misame_raw.index.isin(misame_full_BM_df_filt['SampleID'])]
#misame_raw.reset_index(inplace=True)

#draw from unif distribution with low=min(table)/10 and high=min(table)
random.seed(5)
misame_array = misame_raw.values
zero_len=len(misame_array[pd.isna(misame_array)])
min_value=np.nanmin(misame_array)
rand_array=np.random.uniform(low=min_value/10, high=min_value, size=zero_len)
misame_array[pd.isna(misame_array)]=rand_array
misame_preproc=pd.DataFrame(misame_array, index=misame_raw.index, columns=misame_raw.columns).reset_index()
print(misame_preproc.shape)

loading data
(843, 37072)


In [5]:
print("reading keys")
# Subsetting the original key: remove NAs, sort by rt_difference, remove duplicates from misame and vital ID columns with higher rt_difference 
cross_study_key_original_besthits=pd.read_csv(f"{misame_data_path}/key/IMiC_alignment.csv", sep=',')
cross_study_key_original_besthits=cross_study_key_original_besthits.dropna(subset=['mtb_id_MISAME3', 
                                                                                   'mtb_id_CHILD_ELICIT_VITAL']).sort_values(by='rt_difference')[['mtb_id_MISAME3', 
                                                                                                                                                  'mtb_id_CHILD_ELICIT_VITAL']]
cross_study_key_original_besthits=cross_study_key_original_besthits.drop_duplicates(subset='mtb_id_MISAME3', keep="first")
cross_study_key_original_besthits=cross_study_key_original_besthits.drop_duplicates(subset='mtb_id_CHILD_ELICIT_VITAL', keep="first")
cross_study_key_original_besthits

# Identifying 1-1 overlap IDs from original key
Misame_MisVit2_common_IDs=cross_study_key_original_besthits['mtb_id_MISAME3'].tolist()
Vital_MisVit2_common_IDs=cross_study_key_original_besthits['mtb_id_CHILD_ELICIT_VITAL'].tolist()
MisVit2_rename_dict=dict(zip(cross_study_key_original_besthits['mtb_id_CHILD_ELICIT_VITAL'], cross_study_key_original_besthits['mtb_id_MISAME3']))

misame_1to1_preproc=misame_preproc[['sample_ID'] + Misame_MisVit2_common_IDs]

#save
#misame_1to1_preproc.to_csv(f"{misame_untargeted_out}/misame_untargeted_all.csv", index=False)

reading keys


In [6]:
#load top 104 untargeted metabolites (optional)
top_untargeted_metab = pd.read_excel(f'{metab_id_path}/IMiC_Azad_Metabolite_ID_Share.xlsx', index_col=0)
#keep only the top 104 untargeted metabolites
misame_1to1_preproc.set_index('sample_ID', inplace=True)
untargeted = misame_1to1_preproc[top_untargeted_metab.index]

#save
#untargeted.to_csv(f"{misame_untargeted_out}/misame_untargeted_sapient.csv", index=True)